In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib widget

import sys
import gc
import time
import torch.multiprocessing as mp
import torch.distributed as dist
import os

sys.path.insert(0, "../../src")

import h5py
import matplotlib.pyplot as plt
import numpy as np
import torch
import zarr as z

from juart.dl.model.dc import DataConsistency
from juart.recon.sense import SENSE

from juart.conopt.functional.fourier import (
    fourier_transform_adjoint,
    fourier_transform_forward,
    nonuniform_fourier_transform_adjoint,
)

from juart.conopt.functional.fourier import nonuniform_fourier_transform_adjoint
from juart.conopt.tfs.fourier import nonuniform_transfer_function

from juart.vis.interactive import InteractiveFigure3D, InteractiveMultiPlotter3D

In [ ]:
def setup_DC(
    data: dict,
    shape: tuple[int],
    axes: tuple[int] = (1,2,3),
    device="cpu",
    verbose=True,
    niter: int = 10
):
    dc_block = DataConsistency(
        shape,
        axes = (1,2,3),
        lamda_start=1e-9,
        device=device,
        verbose = True,
        niter = niter
    )

    dc_block.init(
        data["images_regridded"],
        data["kspace_trajectory"],
        sensitivity_maps=data["sensitivity_maps"],
        kspace_mask = data["kspace_mask_source"]
    )

    return dc_block

In [ ]:
dist.init_process_group(
    backend="gloo", init_method="tcp://127.0.0.1:12446", world_size=1, rank=0
)

In [ ]:
saving = True
device = 'cuda:2'
dtype = torch.complex64
niter = 50

# directory = "fibo_phantom_128spk_R8_1191"
store = z.open("/home/jovyan/datasets/fibo_phantom_128spk_R8_1597points")

nX, nY, nZ, nTI, nTE = 128, 128, 128, 1, 1
C = torch.from_numpy(np.array(store["C"]))
k = torch.from_numpy(np.array(store["k"]))[...,None,None]
d = torch.from_numpy(np.array(store["d"]))[...,None,None]


# data_path = "/home/jovyan/datasets/meas_MID00149_FID31870_Kooshball_GRE_30000_preproc-2.h5"
# with h5py.File(data_path, "r") as f:

#     nX, nY, nZ, nTI, nTE = 128, 128, 128, 1, 1
#     k = torch.from_numpy(f["us_10"]['traj'][:])[...,None,None]
#     k = k.reshape([k.shape[0],k.shape[1]*k.shape[2],k.shape[-2],k.shape[-1]])

#     C = torch.from_numpy(f['sens_maps'][:])
#     C = torch.permute(C, (3,0,1,2))

#     d = torch.from_numpy(f["us_10"]['data'][:])[...,None,None]
#     d = d.reshape([d.shape[3],d.shape[1]*d.shape[2],d.shape[-2],d.shape[-1]])

#     print(f"Coilsensitivity shape {C.shape}")
#     print(f"Trajectory shape {k.shape}")
#     print(f"Signal shape {d.shape}")

k /= (2*k.max())
k = k.view(*k.shape, *([1]*(4-k.dim())))

d /= d.abs().max()
d = d.view(*d.shape, *([1]*(4-d.dim())))

C /= C.abs().max()

shape = (nX, nY, nZ, nTI, nTE)

print(k.shape,k.min(),k.max())
print(d.shape,d.real.min(),d.real.max())
print(C.shape,C.real.min(),C.real.max())

kspace_mask_source = torch.randint(0,2,(1, k.shape[1], 1,1))
kspace_mask_target = 1 - kspace_mask_source

AHd = nonuniform_fourier_transform_adjoint(k, d, (nX, nY, nZ))
AHd = torch.sum(torch.conj(C[..., None,None]) * AHd, dim=0)

data ={
        "images_regridded": AHd.to(device),
        "kspace_trajectory": k.to(device),
        "sensitivity_maps": C.to(device),
        "kspace_mask_source": kspace_mask_source.to(device),
        "kspace_mask_target": kspace_mask_target.to(device),
        "kspace_data": d.to(device),
}

In [ ]:
dc_image = torch.zeros_like(data['images_regridded']).detach().clone().to(device)

dc_block = setup_DC(data, shape, (1, 2, 3), device=device, niter=niter)

with torch.no_grad():
    for _ in range(0,1,1):
        dc_image = dc_block(dc_image).to(device)

In [ ]:
dc_image = dc_image[:,:,:,0,0].cpu().abs() / dc_image[:,:,60,0,0].cpu().abs().max()

In [ ]:
InteractiveMultiPlotter3D(
    [dc_image.cpu().abs()],
    title = ["DC"],
    vmin=0,
    vmax=1,
    activate_colorbar=False,
    cmap="gray",
).interactive

In [ ]:
if saving:
    if not os.path.isdir(f"/home/jovyan/images/num_cgsense_reco/{directory}"):
        os.makedirs(f"/home/jovyan/images/num_cgsense_reco/{directory}")

    torch.save(dc_image, f'/home/jovyan/images/num_cgsense_reco/{directory}/{niter}i')
    print("image saved successfully")